# Лабораторная работа №1: Нелинейная регрессия (BatchNorm и Dropout)

**Цель:** исследовать, как размер батча, BatchNorm и Dropout влияют на качество аппроксимации нелинейных функций MLP-моделью.

Что делаем:
1. Генерируем 3 случайные нелинейные регрессии (разные функции).
2. **Часть 1:** тестируем только BatchNorm с 3 размерами батча.
3. **Часть 2:** берем лучший batch size и сравниваем 4 режима:
   - без BatchNorm и Dropout,
   - только BatchNorm,
   - только Dropout,
   - BatchNorm + Dropout.
4. Строим графики и формулируем выводы.


## Фиксированные гиперпараметры (для обеих частей)

- Архитектура MLP: `1 -> 128 -> 64 -> 1` (3 линейных слоя)
- Активация: `ReLU`
- Функция потерь: `MSELoss`
- Оптимизатор: `Adam`
- Learning rate: `1e-3`
- Количество эпох: `250`
- Dropout probability (если используется): `0.2`


In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style='whitegrid')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# 3 разные нелинейные функции

def f1(x):
    return np.sin(1.7 * x) + 0.15 * x**2

def f2(x):
    return np.cos(0.8 * x) * np.exp(-0.05 * x**2) + 0.3 * x

def f3(x):
    return 0.5 * np.sin(2.5 * x) + 0.2 * np.cos(0.5 * x) + 0.03 * x**3

FUNCTIONS = {
    'f1': f1,
    'f2': f2,
    'f3': f3,
}

@dataclass
class DatasetConfig:
    n_points: int = 600
    x_min: float = -4.0
    x_max: float = 4.0
    noise_std: float = 0.15

cfg_data = DatasetConfig()


def make_regression_data(func, cfg: DatasetConfig, seed: int = 42):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(cfg.x_min, cfg.x_max, cfg.n_points))
    y_true = func(x)
    y_noisy = y_true + rng.normal(0.0, cfg.noise_std, size=cfg.n_points)

    x = x.reshape(-1, 1).astype(np.float32)
    y_true = y_true.reshape(-1, 1).astype(np.float32)
    y_noisy = y_noisy.reshape(-1, 1).astype(np.float32)
    return x, y_true, y_noisy


def split_data(x, y, train_frac=0.7, val_frac=0.15):
    n = len(x)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    x_train, y_train = x[:n_train], y[:n_train]
    x_val, y_val = x[n_train:n_train+n_val], y[n_train:n_train+n_val]
    x_test, y_test = x[n_train+n_val:], y[n_train+n_val:]

    return (x_train, y_train), (x_val, y_val), (x_test, y_test)


In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, use_batchnorm: bool = False, dropout_p: float = 0.0):
        super().__init__()

        layers = []
        layers += [nn.Linear(1, 128)]
        if use_batchnorm:
            layers += [nn.BatchNorm1d(128)]
        layers += [nn.ReLU()]
        if dropout_p > 0:
            layers += [nn.Dropout(dropout_p)]

        layers += [nn.Linear(128, 64)]
        if use_batchnorm:
            layers += [nn.BatchNorm1d(64)]
        layers += [nn.ReLU()]
        if dropout_p > 0:
            layers += [nn.Dropout(dropout_p)]

        layers += [nn.Linear(64, 1)]

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@dataclass
class TrainConfig:
    lr: float = 1e-3
    epochs: int = 250

cfg_train = TrainConfig()
criterion = nn.MSELoss()


def to_loader(x, y, batch_size, shuffle=True):
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def eval_mse(model, loader):
    model.eval()
    losses = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        losses.append(loss.item())
    return float(np.mean(losses))


def train_model(x_train, y_train, x_val, y_val, use_batchnorm, dropout_p, batch_size, seed=42):
    set_seed(seed)

    model = MLPRegressor(use_batchnorm=use_batchnorm, dropout_p=dropout_p).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg_train.lr)

    train_loader = to_loader(x_train, y_train, batch_size=batch_size, shuffle=True)
    val_loader = to_loader(x_val, y_val, batch_size=batch_size, shuffle=False)

    history = {'train_mse': [], 'val_mse': []}
    best_state = None
    best_val = float('inf')

    for _ in range(cfg_train.epochs):
        model.train()
        batch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_mse = float(np.mean(batch_losses))
        val_mse = eval_mse(model, val_loader)

        history['train_mse'].append(train_mse)
        history['val_mse'].append(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def predict_numpy(model, x_np):
    model.eval()
    x_t = torch.from_numpy(x_np).to(device)
    y_t = model(x_t).cpu().numpy()
    return y_t


## Часть 1. Только BatchNorm, разные batch size

Тестируем `batch_size in {16, 32, 64}` при `use_batchnorm=True`, `dropout=0.0`.


In [ ]:
batch_sizes = [16, 32, 64]
part1_rows = []

for i, (fname, func) in enumerate(FUNCTIONS.items()):
    x, y_true, y_noisy = make_regression_data(func, cfg_data, seed=100 + i)
    (x_tr, y_tr), (x_val, y_val), (x_te, y_te) = split_data(x, y_noisy)

    for bs in batch_sizes:
        model, hist = train_model(
            x_tr, y_tr, x_val, y_val,
            use_batchnorm=True,
            dropout_p=0.0,
            batch_size=bs,
            seed=42,
        )

        test_loader = to_loader(x_te, y_te, batch_size=bs, shuffle=False)
        test_mse = eval_mse(model, test_loader)
        val_best = float(np.min(hist['val_mse']))

        part1_rows.append({
            'function': fname,
            'batch_size': bs,
            'val_mse_best': val_best,
            'test_mse': test_mse,
        })

part1_df = pd.DataFrame(part1_rows)
part1_df


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=part1_df, x='batch_size', y='test_mse', hue='function')
plt.title('Часть 1: test MSE для разных batch size (только BatchNorm)')
plt.tight_layout()
plt.show()

avg_by_bs = part1_df.groupby('batch_size', as_index=False)['test_mse'].mean().sort_values('test_mse')
avg_by_bs


In [ ]:
best_batch_size = int(avg_by_bs.iloc[0]['batch_size'])
print('Лучший batch size по среднему test MSE:', best_batch_size)


## Часть 2. Сравнение 4 режимов регуляризации при лучшем batch size

Режимы:
1. `no_bn_no_do` — без BatchNorm и без Dropout
2. `bn_only` — только BatchNorm
3. `do_only` — только Dropout
4. `bn_do` — BatchNorm + Dropout


In [ ]:
regimes = {
    'no_bn_no_do': {'use_batchnorm': False, 'dropout_p': 0.0},
    'bn_only': {'use_batchnorm': True, 'dropout_p': 0.0},
    'do_only': {'use_batchnorm': False, 'dropout_p': 0.2},
    'bn_do': {'use_batchnorm': True, 'dropout_p': 0.2},
}

part2_rows = []
best_models_part2 = {}
data_cache = {}

for i, (fname, func) in enumerate(FUNCTIONS.items()):
    x, y_true, y_noisy = make_regression_data(func, cfg_data, seed=100 + i)
    data_cache[fname] = (x, y_true, y_noisy)
    (x_tr, y_tr), (x_val, y_val), (x_te, y_te) = split_data(x, y_noisy)

    for rname, cfg in regimes.items():
        model, hist = train_model(
            x_tr, y_tr, x_val, y_val,
            use_batchnorm=cfg['use_batchnorm'],
            dropout_p=cfg['dropout_p'],
            batch_size=best_batch_size,
            seed=42,
        )

        test_loader = to_loader(x_te, y_te, batch_size=best_batch_size, shuffle=False)
        test_mse = eval_mse(model, test_loader)
        val_best = float(np.min(hist['val_mse']))

        part2_rows.append({
            'function': fname,
            'regime': rname,
            'val_mse_best': val_best,
            'test_mse': test_mse,
        })

        best_models_part2[(fname, rname)] = model

part2_df = pd.DataFrame(part2_rows)
part2_df


In [ ]:
plt.figure(figsize=(11, 4))
sns.barplot(data=part2_df, x='regime', y='test_mse', hue='function')
plt.title(f'Часть 2: test MSE при batch_size={best_batch_size}')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
best_regime_per_function = (
    part2_df.sort_values('test_mse')
    .groupby('function', as_index=False)
    .first()[['function', 'regime', 'test_mse']]
)
best_regime_per_function


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharey=False)

for ax, (fname, func) in zip(axes, FUNCTIONS.items()):
    x, y_true, y_noisy = data_cache[fname]

    best_regime = best_regime_per_function.loc[
        best_regime_per_function['function'] == fname,
        'regime'
    ].iloc[0]
    model = best_models_part2[(fname, best_regime)]
    y_pred = predict_numpy(model, x)

    order = np.argsort(x[:, 0])
    x_plot = x[order, 0]

    ax.plot(x_plot, y_true[order, 0], label='Исходная функция', linewidth=2)
    ax.scatter(x[:, 0], y_noisy[:, 0], s=10, alpha=0.4, label='Шумные точки')
    ax.plot(x_plot, y_pred[order, 0], label=f'Лучшая аппроксимация ({best_regime})', linewidth=2)

    ax.set_title(f'{fname}: лучшая модель')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
summary_part1 = part1_df.groupby('batch_size', as_index=False)['test_mse'].mean().sort_values('test_mse')
summary_part2 = part2_df.groupby('regime', as_index=False)['test_mse'].mean().sort_values('test_mse')

print('=== Часть 1: средний test MSE по batch size ===')
print(summary_part1.to_string(index=False))

print('\n=== Часть 2: средний test MSE по режимам ===')
print(summary_part2.to_string(index=False))

print('\n=== Лучший режим для каждой функции ===')
print(best_regime_per_function.to_string(index=False))


## Автоматически сформированные выводы

In [ ]:
best_bs = int(summary_part1.iloc[0]['batch_size'])
best_global_regime = summary_part2.iloc[0]['regime']

print(f'1) В части 1 лучший размер батча: {best_bs}.')
print(f'2) В среднем лучший режим регуляризации: {best_global_regime}.')

if best_global_regime == 'bn_only':
    print('3) BatchNorm дал наилучшее качество и стабилизировал обучение.')
elif best_global_regime == 'do_only':
    print('3) Dropout без BatchNorm дал лучшую обобщающую способность.')
elif best_global_regime == 'bn_do':
    print('3) Комбинация BatchNorm + Dropout оказалась наиболее эффективной.')
else:
    print('3) Базовая модель без регуляризации показала лучший средний результат на этих данных.')

print('4) По таблице best_regime_per_function видно, одинаков ли лучший режим для всех функций или зависит от типа нелинейности.')
